### Imports

In [ ]:
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_curve, roc_auc_score, mean_absolute_error

import sys
sys.path.append('..')

from src.utils.dataloading import FeaturesDataset
from src.utils.model import AgeNet
#from src.utils.utils import train, extract_features

# Evaluating the model

## Setup

In [ ]:
# setup
seed = 42
torch.manual_seed(seed)
# device setup
device = (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
print(f"Training on device {device}.")

### loading

In [ ]:
model_path = "../data/model/AgeNet.pt"

model = AgeNet(seq_len=4, d_model=1280, nhead=8)
model.load_state_dict(torch.load(model_path, weights_only=True, map_location=device))
model.backbone.freeze()
model.use_precompute()

In [ ]:
def evaluate(model, dataloader, device):
    model.to(device)
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in dataloader:
            inputs, labels = batch
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return all_preds, all_labels

Since there is a class imbalance between positive examples (persons) and negative examples (background), a model that predicts every sample as a person could still achieve a high accuracy. Therefore, accuracy alone is not a reliable performance metric. Instead, we use the ROC-AUC score to better evaluate the model's ability to distinguish between the two classes.

In [ ]:
auc_score = roc_auc_score()